This notebook substitutes some classes in an experience by "debug" versions of them, which write to file almost every intermidiate step, as to help detect any incoherence in the code

In [ ]:
PATH_TO_STORE_EXPERIMENTS = "data\\rl_training"

In [ ]:
experiment_name = "dqn_multi_agent"

In [ ]:
import shutil
from automl.basic_components.artifact_management import open_or_create_folder
import os

base_experiment_path = "data\\hp_lodaded_exps"

if True : # if we copy the experiment path

    base_original_experiment_path = "C:\\Experiments\\multiwalker_ppo\\HPExperiments\\1\\configuration_125\\1"
    
    experiment_path = open_or_create_folder(base_experiment_path, experiment_name)

    shutil.copytree(base_original_experiment_path, experiment_path, dirs_exist_ok=True)

else:
    base_experiment_path = "data\\hp_other_exps"
    experiment_path = os.path.join(base_experiment_path, "multiwalker_ppo_1_125_1")

# Preparation before loading experiment

In [ ]:
from automl.loggers.global_logger import activate_global_logger

activate_global_logger(experiment_path, {"necessary_logger_level" : "DEBUG"})

## Change logging system

In [ ]:
from automl.loggers.logger_component import LoggerSchema 

LoggerSchema.get_schema_parameter_signature("write_to_file_when_text_lines_over").change_default_value(-1)
LoggerSchema.get_schema_parameter_signature("necessary_logger_level").change_default_value("INFO")

In [ ]:
from automl.loggers.component_with_results import ResultLogger


ResultLogger.get_schema_parameter_signature("save_results_on_log").change_default_value(True)

# The base Experiment

## Base Configuration

In [ ]:
from automl.rl.rl_pipeline import RLPipelineComponent
from automl.utils.file_component_utils import gen_component_from_path


rl_pipeline : RLPipelineComponent = gen_component_from_path(experiment_path)

rl_pipeline.pass_input({"logger_input" : {
    "necessary_logger_level" : "INFO",
    "write_to_file_when_text_lines_over" : -1
    }})

rl_pipeline.pass_input({"results_logger_input" : {
    "save_results_on_log" : True
}})


# Manual Hyperparameter Tuning

In [ ]:
#substitute_value_in_dict(learner_input, "target_update_rate", 0.5511208693081078)

In [ ]:
rl_pipeline.pass_input({"times_to_run" : 1})

# Do the training

In [ ]:
from automl.loggers.global_logger import activate_global_logger

activate_global_logger(rl_pipeline.get_artifact_directory())

In [ ]:
rl_pipeline.run()

## Save configuration

In [ ]:
#rl_pipeline.save_configuration(save_exposed_values=True)
from automl.basic_components.state_management import save_state


save_state(rl_pipeline, save_definition=True)

## See Results

In [ ]:
AGGREGATE_NUMBER = 5

In [ ]:

from automl.loggers.result_logger import ResultLogger
    
results_logger : ResultLogger = rl_pipeline.get_results_logger()

In [ ]:
#results_logger.plot_graph(x_axis='episode', y_axis=[('total_reward', name)], to_show=False)
results_logger.plot_confidence_interval(x_axis='episode', y_column='episode_reward',show_std=True, to_show=False, y_values_label=experiment_name, aggregate_number=AGGREGATE_NUMBER)
results_logger.plot_linear_regression(x_axis='episode', y_axis='episode_reward', to_show=False, y_label=experiment_name + '_linear')
